# 🇮🇳 Mini Hindi-GPT From Scratch
### CSE556 NLP Assignment 2 — Advanced Procedural Implementation
All cells execute sequentially top-to-bottom. No standalone function definitions.

In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sentencepiece', 'tqdm', '--quiet'])
print('Done')

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────
import os, random, math, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| Device:', DEVICE)

In [ ]:
# ── Cell 3: Hyperparameters ───────────────────────────────────────────
VOCAB_SIZE    = 5000
SP_MODEL_PATH = 'hindi_bpe.model'
TRAIN_TXT     = 'corpus_train.txt'

EMBED_DIM     = 256
N_HEADS       = 8
N_LAYERS      = 6
FFN_DIM       = 1024
BLOCK_SIZE    = 256
DROPOUT       = 0.1

LM_BATCH      = 32
LM_EPOCHS     = 5
LM_LR         = 3e-4
WARMUP_STEPS  = 500
GRAD_CLIP     = 1.0
LABEL_SMOOTH  = 0.1

CLS_BATCH     = 64
CLS_EPOCHS    = 5
CLS_LR        = 1e-4
N_CLASSES     = 3

DATA_CORPUS   = './Data/hindi_corpus/train'
DATA_CLS      = './Data/text_classification_dataset/train.csv'
LM_CKPT       = 'lm_checkpoint.pt'
CLS_CKPT      = 'cls_checkpoint.pt'
print('Config set.')

---
## 📦 Task 1 — Data Preprocessing

In [ ]:
# ── Cell 4: Load Hindi corpus from disk ──────────────────────────────
corpus_files = sorted(glob.glob(os.path.join(DATA_CORPUS, '*.txt')))
all_texts = []
for fp in tqdm(corpus_files, desc='Loading corpus'):
    try:
        with open(fp, 'r', encoding='utf-8') as fh:
            t = fh.read().strip()
        if t:
            all_texts.append(t)
    except Exception:
        pass

print(f'Total documents loaded: {len(all_texts):,}')
print('Sample text (first 200 chars):')
print(all_texts[0][:200])

In [ ]:
# ── Cell 5: Train / Val split of corpus ──────────────────────────────
random.shuffle(all_texts)
split_idx   = int(0.9 * len(all_texts))
train_texts = all_texts[:split_idx]
val_texts   = all_texts[split_idx:]
print(f'Train docs: {len(train_texts):,} | Val docs: {len(val_texts):,}')

In [ ]:
# ── Cell 6: Write corpus to file & train SentencePiece BPE ───────────
if not os.path.exists(SP_MODEL_PATH):
    print('Writing training corpus...')
    with open(TRAIN_TXT, 'w', encoding='utf-8') as fh:
        fh.write('\n'.join(train_texts))
    print('Training SentencePiece BPE tokenizer (vocab=5000)...')
    spm.SentencePieceTrainer.train(
        input=TRAIN_TXT,
        model_prefix='hindi_bpe',
        vocab_size=VOCAB_SIZE,
        model_type='bpe',
        character_coverage=0.9995,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3,
        input_sentence_size=500000,
        shuffle_input_sentence=True,
    )
    print('Tokenizer saved.')
else:
    print('Tokenizer already exists, loading...')

In [ ]:
# ── Cell 7: Load tokenizer & test encode / decode ────────────────────
sp     = spm.SentencePieceProcessor()
sp.Load(SP_MODEL_PATH)
PAD_ID = sp.pad_id()   # 0
BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3

sample_text   = 'हिन्दी भाषा में ट्रांसफार्मर मॉडल'
sample_ids    = sp.EncodeAsIds(sample_text)
sample_decode = sp.DecodeIds(sample_ids)
print('Encode:', sample_ids)
print('Decode:', sample_decode)

In [ ]:
# ── Cell 8: Build LM dataset — tokenise all corpus texts ─────────────
class LMDataset(Dataset):
    def __init__(self, texts, block_size):
        token_list = []
        for t in tqdm(texts, desc='Tokenising LM', leave=False):
            token_list += [BOS_ID] + sp.EncodeAsIds(t) + [EOS_ID]
        self.data  = torch.tensor(token_list, dtype=torch.long)
        self.block = block_size
        print(f'  Total tokens: {len(self.data):,}')
    def __len__(self):
        return max(0, len(self.data) - self.block)
    def __getitem__(self, i):
        chunk = self.data[i : i + self.block + 1]
        return chunk[:-1], chunk[1:]

print('Building train LM dataset...')
lm_train_ds = LMDataset(train_texts, BLOCK_SIZE)
print('Building val LM dataset...')
lm_val_ds   = LMDataset(val_texts,   BLOCK_SIZE)

lm_train_loader = DataLoader(lm_train_ds, batch_size=LM_BATCH, shuffle=True,  num_workers=0, pin_memory=(DEVICE=='cuda'))
lm_val_loader   = DataLoader(lm_val_ds,   batch_size=LM_BATCH, shuffle=False, num_workers=0, pin_memory=(DEVICE=='cuda'))
print(f'Train batches: {len(lm_train_loader):,} | Val batches: {len(lm_val_loader):,}')

In [ ]:
# ── Cell 9: Load classification CSV & inspect ─────────────────────────
df = pd.read_csv(DATA_CLS)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print(df.head(3))
print('Unique labels:', df.iloc[:, -1].unique())

In [ ]:
# ── Cell 10: Map labels → 0 / 1 / 2 and split 90/10 ─────────────────
text_col  = df.columns[0]
label_col = df.columns[-1]

raw_labels = df[label_col].astype(str).str.strip().str.lower()
unique_sorted = sorted(raw_labels.unique())
# Auto-map: sort alphabetically → assign 0,1,2
# Override for common sentiment keywords
label_map = {}
neg_kw = {'negative','neg','bad','0','नकारात्मक'}
pos_kw = {'positive','pos','good','2','सकारात्मक'}
for lbl in unique_sorted:
    if lbl in neg_kw:   label_map[lbl] = 0
    elif lbl in pos_kw: label_map[lbl] = 2
    else:               label_map[lbl] = 1
# Ensure exactly 0,1,2 used
vals = sorted(set(label_map.values()))
remap = {v: i for i, v in enumerate(vals)}
label_map = {k: remap[v] for k,v in label_map.items()}
print('Label map:', label_map)

df = df.copy()
df['label_id'] = raw_labels.map(label_map)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)
print('Class distribution:\n', df['label_id'].value_counts())

df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
cls_split    = int(0.9 * len(df))
cls_train_df = df.iloc[:cls_split].reset_index(drop=True)
cls_val_df   = df.iloc[cls_split:].reset_index(drop=True)
print(f'Cls train: {len(cls_train_df):,} | Cls val: {len(cls_val_df):,}')

In [ ]:
# ── Cell 11: Classification dataset & padded DataLoader ──────────────
class ClsDataset(Dataset):
    def __init__(self, dataframe, txt_col, block_size):
        self.texts  = dataframe[txt_col].astype(str).tolist()
        self.labels = dataframe['label_id'].tolist()
        self.block  = block_size
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        ids = sp.EncodeAsIds(self.texts[i])[:self.block]
        return torch.tensor(ids, dtype=torch.long), self.labels[i]

def pad_collate(batch):
    seqs, labs = zip(*batch)
    max_len = max(len(s) for s in seqs)
    padded  = torch.full((len(seqs), max_len), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    return padded, torch.tensor(labs, dtype=torch.long)

cls_train_ds = ClsDataset(cls_train_df, text_col, BLOCK_SIZE)
cls_val_ds   = ClsDataset(cls_val_df,   text_col, BLOCK_SIZE)
cls_train_loader = DataLoader(cls_train_ds, batch_size=CLS_BATCH, shuffle=True,  collate_fn=pad_collate, num_workers=0)
cls_val_loader   = DataLoader(cls_val_ds,   batch_size=CLS_BATCH, shuffle=False, collate_fn=pad_collate, num_workers=0)
print(f'Cls train batches: {len(cls_train_loader)} | Val batches: {len(cls_val_loader)}')

---
## 🤖 Task 2 — Decoder-Only Transformer + Language Modelling

In [ ]:
# ── Cell 12: Multi-Head Causal Self-Attention ─────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, dropout):
        super().__init__()
        assert embed_dim % n_heads == 0
        self.n_heads   = n_heads
        self.head_dim  = embed_dim // n_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.proj      = nn.Linear(embed_dim, embed_dim,     bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.out_drop  = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        # Project → Q, K, V
        q, k, v = self.qkv(x).split(C, dim=2)                         # each (B,T,C)
        # Reshape to heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B,H,T,d)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        # Scaled dot-product attention
        scores = (q @ k.transpose(-2, -1)) * self.scale                # (B,H,T,T)
        # Causal mask — lower-triangular so token i only sees tokens ≤ i
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        scores = scores.masked_fill(~causal_mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)                            # (B,H,T,T)
        weights = self.attn_drop(weights)
        out = weights @ v                                              # (B,H,T,d)
        out = out.transpose(1, 2).contiguous().view(B, T, C)          # (B,T,C)
        return self.out_drop(self.proj(out))

print('MultiHeadAttention defined.')

In [ ]:
# ── Cell 13: FeedForward Network ──────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ffn_dim, dropout):
        super().__init__()
        self.fc1  = nn.Linear(embed_dim, ffn_dim)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(ffn_dim, embed_dim)
        self.out_drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.out_drop(x)
        return x

print('FeedForward defined.')

In [ ]:
# ── Cell 14: Transformer Block (Pre-LayerNorm + residual) ────────────
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1  = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, dropout)
        self.ln2  = nn.LayerNorm(embed_dim)
        self.ffn  = FeedForward(embed_dim, ffn_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # attention residual
        x = x + self.ffn(self.ln2(x))    # FFN residual
        return x

print('TransformerBlock defined.')

In [ ]:
# ── Cell 15: MiniGPT Backbone ─────────────────────────────────────────
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, n_heads, n_layers, ffn_dim, dropout):
        super().__init__()
        self.tok_emb    = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.pos_emb    = nn.Embedding(block_size, embed_dim)
        self.drop       = nn.Dropout(dropout)
        self.blocks     = nn.Sequential(
            *[TransformerBlock(embed_dim, n_heads, ffn_dim, dropout) for _ in range(n_layers)]
        )
        self.ln_f       = nn.LayerNorm(embed_dim)
        self.block_size = block_size
        # GPT-2 style weight init
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device).unsqueeze(0)   # (1,T)
        x    = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x    = self.blocks(x)
        x    = self.ln_f(x)
        return x   # (B, T, embed_dim)

print('MiniGPT backbone defined.')

In [ ]:
# ── Cell 16: GPTLanguageModel (backbone + LM head + weight tying) ────
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size, n_heads, n_layers, ffn_dim, dropout):
        super().__init__()
        self.backbone = MiniGPT(vocab_size, embed_dim, block_size,
                                n_heads, n_layers, ffn_dim, dropout)
        self.lm_head  = nn.Linear(embed_dim, vocab_size, bias=False)
        # Weight tying: share token-embedding weights with LM head (fewer params + better perf)
        self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, idx, targets=None, label_smoothing=0.0):
        x      = self.backbone(idx)          # (B, T, C)
        logits = self.lm_head(x)             # (B, T, V)
        loss   = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.view(B * T, V),
                targets.view(B * T),
                ignore_index=PAD_ID,
                label_smoothing=label_smoothing,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            ctx    = idx[:, -self.backbone.block_size:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :] / temperature
            # top-k filtering
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, [-1]]] = float('-inf')
            probs  = F.softmax(logits, dim=-1)
            nxt    = torch.multinomial(probs, num_samples=1)
            idx    = torch.cat([idx, nxt], dim=1)
        return idx

lm_model = GPTLanguageModel(
    vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, block_size=BLOCK_SIZE,
    n_heads=N_HEADS, n_layers=N_LAYERS, ffn_dim=FFN_DIM, dropout=DROPOUT
).to(DEVICE)
print(f'GPTLanguageModel | Params: {sum(p.numel() for p in lm_model.parameters()):,}')

In [ ]:
# ── Cell 17: Optimizer + Cosine LR Scheduler with Warmup ─────────────
lm_optimizer  = torch.optim.AdamW(
    lm_model.parameters(), lr=LM_LR, weight_decay=0.01, betas=(0.9, 0.95)
)
total_steps = LM_EPOCHS * len(lm_train_loader)

# Lambda: linear warmup then cosine decay down to 10% of peak LR
min_lr_ratio = 0.1
def lr_lambda(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    cosine   = 0.5 * (1 + math.cos(math.pi * progress))
    return min_lr_ratio + (1 - min_lr_ratio) * cosine

lm_scheduler  = torch.optim.lr_scheduler.LambdaLR(lm_optimizer, lr_lambda)
print(f'Optimizer ready. Total training steps: {total_steps:,}')

In [ ]:
# ── Cell 18: Language Model Training Loop ─────────────────────────────
lm_train_losses = []
lm_val_losses   = []
best_lm_val     = float('inf')

for epoch in range(1, LM_EPOCHS + 1):
    # ─ Train phase ─
    lm_model.train()
    running_loss = 0.0
    num_batches  = 0
    pbar = tqdm(lm_train_loader, desc=f'Epoch {epoch}/{LM_EPOCHS} [LM Train]')
    for x_batch, y_batch in pbar:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        lm_optimizer.zero_grad()
        _, loss  = lm_model(x_batch, y_batch, label_smoothing=LABEL_SMOOTH)
        loss.backward()
        nn.utils.clip_grad_norm_(lm_model.parameters(), GRAD_CLIP)
        lm_optimizer.step()
        lm_scheduler.step()
        running_loss += loss.item()
        num_batches  += 1
        pbar.set_postfix(loss=f'{running_loss/num_batches:.4f}',
                         lr=f'{lm_scheduler.get_last_lr()[0]:.2e}')
    train_loss = running_loss / num_batches

    # ─ Val phase ─
    lm_model.eval()
    val_loss_sum = 0.0
    val_batches  = 0
    with torch.no_grad():
        for x_v, y_v in tqdm(lm_val_loader, desc=f'Epoch {epoch} [LM Val]', leave=False):
            x_v, y_v = x_v.to(DEVICE), y_v.to(DEVICE)
            _, vloss = lm_model(x_v, y_v)
            val_loss_sum += vloss.item()
            val_batches  += 1
    val_loss   = val_loss_sum / max(val_batches, 1)
    perplexity = math.exp(val_loss)

    lm_train_losses.append(train_loss)
    lm_val_losses.append(val_loss)
    print(f'Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Perplexity: {perplexity:.2f}')

    if val_loss < best_lm_val:
        best_lm_val = val_loss
        torch.save(lm_model.state_dict(), LM_CKPT)
        print(f'  ✔ Best LM checkpoint saved  (val_loss={val_loss:.4f})')

print('LM training complete.')

In [ ]:
# ── Cell 19: Plot LM Loss & Perplexity Curves ─────────────────────────
epochs_ax = list(range(1, len(lm_train_losses) + 1))
perps     = [math.exp(l) for l in lm_val_losses]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_ax, lm_train_losses, 'o-', color='steelblue', label='Train Loss')
axes[0].plot(epochs_ax, lm_val_losses,   's--', color='tomato',  label='Val Loss')
axes[0].set_title('Language Model — Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_ax, perps, 'd-', color='darkorange')
axes[1].set_title('Validation Perplexity')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')

plt.tight_layout()
plt.savefig('lm_training_curves.png', dpi=150)
plt.show()
print(f'Best Val Perplexity: {min(perps):.2f}')

In [ ]:
# ── Cell 20: Load best checkpoint & generate 50-token Hindi text ──────
lm_model.load_state_dict(torch.load(LM_CKPT, map_location=DEVICE))
lm_model.eval()

prompts = ['भारत एक', 'हिन्दी भाषा', 'विज्ञान और तकनीक']
gen_output_lines = []

print('=== Generated Hindi Text (50 new tokens each) ===')
for prompt_str in prompts:
    prompt_ids = torch.tensor([sp.EncodeAsIds(prompt_str)], dtype=torch.long).to(DEVICE)
    out_ids    = lm_model.generate(prompt_ids, max_new_tokens=50, temperature=0.8, top_k=50)
    out_text   = sp.DecodeIds(out_ids[0].tolist())
    print(f'\nPrompt: "{prompt_str}"')
    print('Output:', out_text)
    gen_output_lines.append(f'Prompt: {prompt_str}\nOutput: {out_text}')

with open('generated_text.txt', 'w', encoding='utf-8') as fh:
    fh.write('\n\n'.join(gen_output_lines))
print('\n✔ generated_text.txt saved.')

---
## 🎯 Task 3 — Text Classification with GPT Backbone

In [ ]:
# ── Cell 21: GPTClassifier — reuse backbone, add linear head ─────────
class GPTClassifier(nn.Module):
    def __init__(self, lm_model_ref, n_classes, dropout=0.1):
        super().__init__()
        self.backbone  = lm_model_ref.backbone
        embed_dim      = self.backbone.ln_f.normalized_shape[0]
        self.cls_drop  = nn.Dropout(dropout)
        self.cls_head  = nn.Linear(embed_dim, n_classes)
        nn.init.normal_(self.cls_head.weight, std=0.02)
        nn.init.zeros_(self.cls_head.bias)

    def forward(self, idx, labels=None):
        x       = self.backbone(idx)                      # (B, T, C)
        # Causal model → use LAST non-padding token for classification
        mask    = (idx != PAD_ID).long()                  # (B, T)
        last_pos = (mask.sum(dim=1) - 1).clamp(min=0)    # (B,)
        feat    = x[torch.arange(x.size(0), device=x.device), last_pos]  # (B, C)
        feat    = self.cls_drop(feat)
        logits  = self.cls_head(feat)                     # (B, n_classes)
        loss    = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return logits, loss

# Load best LM weights into lm_model before building classifier
lm_model.load_state_dict(torch.load(LM_CKPT, map_location=DEVICE))
cls_model = GPTClassifier(lm_model, N_CLASSES, dropout=DROPOUT).to(DEVICE)
print(f'GPTClassifier | Params: {sum(p.numel() for p in cls_model.parameters()):,}')

In [ ]:
# ── Cell 22: Classifier Optimizer (differential learning rates) ───────
backbone_params = list(cls_model.backbone.parameters())
head_params     = list(cls_model.cls_head.parameters())
cls_optimizer   = torch.optim.AdamW([
    {'params': backbone_params, 'lr': CLS_LR * 0.1},   # lower LR for pretrained backbone
    {'params': head_params,     'lr': CLS_LR},          # higher LR for new head
], weight_decay=0.01)

cls_total_steps = CLS_EPOCHS * len(cls_train_loader)
def cls_lr_lambda(step):
    warmup = 100
    if step < warmup:
        return step / max(1, warmup)
    progress = (step - warmup) / max(1, cls_total_steps - warmup)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

cls_scheduler = torch.optim.lr_scheduler.LambdaLR(cls_optimizer, cls_lr_lambda)
print(f'Cls optimizer ready. Steps: {cls_total_steps:,}')

In [ ]:
# ── Cell 23: Classifier Training Loop ────────────────────────────────
cls_train_losses = []
cls_val_accs     = []
best_cls_acc     = 0.0

for epoch in range(1, CLS_EPOCHS + 1):
    # ─ Train phase ─
    cls_model.train()
    running_loss = 0.0
    num_batches  = 0
    pbar = tqdm(cls_train_loader, desc=f'Epoch {epoch}/{CLS_EPOCHS} [CLS Train]')
    for x_batch, y_batch in pbar:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        cls_optimizer.zero_grad()
        _, loss  = cls_model(x_batch, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(cls_model.parameters(), GRAD_CLIP)
        cls_optimizer.step()
        cls_scheduler.step()
        running_loss += loss.item()
        num_batches  += 1
        pbar.set_postfix(loss=f'{running_loss/num_batches:.4f}')
    train_loss = running_loss / num_batches

    # ─ Val phase ─
    cls_model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for x_v, y_v in tqdm(cls_val_loader, desc=f'Epoch {epoch} [CLS Val]', leave=False):
            x_v, y_v = x_v.to(DEVICE), y_v.to(DEVICE)
            logits, _ = cls_model(x_v)
            preds     = logits.argmax(dim=-1)
            correct  += (preds == y_v).sum().item()
            total    += y_v.size(0)
    val_acc = correct / max(total, 1)

    cls_train_losses.append(train_loss)
    cls_val_accs.append(val_acc)
    print(f'Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%')

    if val_acc > best_cls_acc:
        best_cls_acc = val_acc
        torch.save(cls_model.state_dict(), CLS_CKPT)
        print(f'  ✔ Best classifier saved  (acc={val_acc*100:.2f}%)')

print(f'\nBest Val Accuracy: {best_cls_acc*100:.2f}%')

In [ ]:
# ── Cell 24: Plot Classifier Training Curves ──────────────────────────
epochs_ax = list(range(1, len(cls_train_losses) + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_ax, cls_train_losses, 'o-', color='steelblue')
axes[0].set_title('Classifier — Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs_ax, [a * 100 for a in cls_val_accs], 's-', color='seagreen')
axes[1].set_title('Classifier — Val Accuracy (%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')

plt.tight_layout()
plt.savefig('cls_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 25: Load best classifier & show 5 correct predictions ───────
cls_model.load_state_dict(torch.load(CLS_CKPT, map_location=DEVICE))
cls_model.eval()

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
correct_samples = []

with torch.no_grad():
    for idx_s in range(len(cls_val_ds)):
        ids_s, true_lbl = cls_val_ds[idx_s]
        ids_s    = ids_s.unsqueeze(0).to(DEVICE)
        logits_s, _ = cls_model(ids_s)
        pred_lbl = logits_s.argmax(dim=-1).item()
        if pred_lbl == true_lbl:
            review_text = cls_val_df[text_col].iloc[idx_s]
            correct_samples.append(
                f"[{label_names[true_lbl]}] {str(review_text)[:250]}"
            )
        if len(correct_samples) >= 5:
            break

print('=== 5 Correctly Predicted Reviews ===')
for i, s in enumerate(correct_samples, 1):
    print(f'{i}. {s}\n')

with open('cls_sample_outputs.txt', 'w', encoding='utf-8') as fh:
    fh.write('\n\n'.join(correct_samples))
print('✔ cls_sample_outputs.txt saved.')

In [ ]:
# ── Cell 26: Final Results Summary ───────────────────────────────────
best_ppl = math.exp(min(lm_val_losses)) if lm_val_losses else float('nan')
best_acc = max(cls_val_accs) * 100      if cls_val_accs  else float('nan')
n_params = sum(p.numel() for p in lm_model.parameters())

print('=' * 60)
print('         MINI HINDI-GPT — FINAL RESULTS')
print('=' * 60)
print(f'  Tokenizer    : SentencePiece BPE  | Vocab : {VOCAB_SIZE}')
print(f'  Model        : {N_LAYERS}-Layer Decoder-Only GPT-2 style')
print(f'  Parameters   : {n_params:,}')
print(f'  Embed / Heads: {EMBED_DIM} / {N_HEADS}  | Block : {BLOCK_SIZE}')
print('-' * 60)
print(f'  [Task 2] Best Val Perplexity  : {best_ppl:.2f}')
print(f'  [Task 3] Best Val Accuracy    : {best_acc:.2f}%')
print('=' * 60)
print('  Output files:')
print(f'    {LM_CKPT}          LM weights')
print(f'    {CLS_CKPT}         Classifier weights')
print('    generated_text.txt         Hindi text generation')
print('    cls_sample_outputs.txt     5 correct review predictions')
print('    lm_training_curves.png     LM loss/perplexity plots')
print('    cls_training_curves.png    Classifier accuracy plot')